# 01 - Data Ingestion & Preprocessing Pipeline
**ExxonMobil DataWorks Challenge 2026**  
**Component Architecture (Option B):** 3-Parquet Relational Data Layer

---

### Objectives
1. Ingest raw competition workbooks from `TestCases/`:
   - `01 Introduction.xlsx` (Uncertainty parameters, nomenclature)
   - `02 Field Production.xlsx` (Field observation rates, injection, and 100 simulation curves)
2. Generate 3 clean, compressed, read-only Parquet files in `data/`:
   - **`cases_metadata.parquet`**: 100 rows containing the 4 uncertainty multipliers and competition-aligned train/val/test splits + 5-fold CV tags.
   - **`field_history.parquet`**: Ground-truth historical production (`RAMP_History`) + observed quarterly field rates and water injection (1998–2008).
   - **`simulation_timeseries.parquet`**: All 100 simulation cases (Gas, Oil, Water rates & cumulatives) in clean tabular format with quarterly survey alignment.
3. Validate data integrity, non-null guarantees, and monotonicity.

In [1]:
import os
import time
from pathlib import Path
import pandas as pd
import numpy as np

# Dynamically locate project root
current_dir = Path.cwd().resolve()
root = current_dir
while not (root / 'TestCases').exists() and root != root.parent:
    root = root.parent

DATA_RAW = root / 'TestCases'
DATA_OUT = root / 'data'
DATA_OUT.mkdir(exist_ok=True, parents=True)

print(f"Project root: {root}")
print(f"Raw data path: {DATA_RAW}")
print(f"Output data path: {DATA_OUT}")

Project root: /Users/tengkuanas/Projects/mwap-exdw-surrogate-model
Raw data path: /Users/tengkuanas/Projects/mwap-exdw-surrogate-model/TestCases
Output data path: /Users/tengkuanas/Projects/mwap-exdw-surrogate-model/data


## 1. Cases Metadata & Uncertainty Parameters
Extracts `Fault Transmissibility`, `Porosity Multiplier`, `Permeability Multiplier`, and `Aquifer Pore Volume` for all 100 cases.  
Assigns:
- **`split_type`**: `train` (Cases 1–70), `val` (Cases 71–85), `test` (Cases 86–100)
- **`fold`**: 5-Fold GroupKFold indices (0–4) for Cases 1–70 as specified in `DATA_CONTRACT.md`.

In [2]:
t0 = time.time()
print("Ingesting 01 Introduction.xlsx (Uncertainty Values)...")

df_unc = pd.read_excel(DATA_RAW / '01 Introduction.xlsx', sheet_name='Uncertainty Values')
df_unc.rename(columns={
    'Item': 'case_name',
    'Fault Transmissibility': 'fault_transmissibility',
    'Porosity Multiplier': 'porosity_multiplier',
    'Permeability Multiplier': 'permeability_multiplier',
    'Aquifer Pore Volume': 'aquifer_pore_volume'
}, inplace=True)

df_unc['case_id'] = df_unc['case_name'].str.extract(r'(\d+)').astype(int)

# Apply competition split tags
df_unc['split_type'] = np.where(df_unc['case_id'] <= 70, 'train',
                        np.where(df_unc['case_id'] <= 85, 'val', 'test'))

# 5-fold CV assignment for train cases (1..70)
df_unc['fold'] = -1
train_mask = df_unc['split_type'] == 'train'
df_unc.loc[train_mask, 'fold'] = (df_unc.loc[train_mask, 'case_id'] - 1) % 5

cols_cases = [
    'case_id', 'case_name', 'split_type', 'fold',
    'fault_transmissibility', 'porosity_multiplier',
    'permeability_multiplier', 'aquifer_pore_volume'
]
df_cases = df_unc[cols_cases].sort_values('case_id').reset_index(drop=True)

out_cases = DATA_OUT / 'cases_metadata.parquet'
df_cases.to_parquet(out_cases, index=False)

print(f"Saved: {out_cases} (Shape: {df_cases.shape}, Time: {time.time()-t0:.2f}s)")
df_cases.head()

Ingesting 01 Introduction.xlsx (Uncertainty Values)...
Saved: /Users/tengkuanas/Projects/mwap-exdw-surrogate-model/data/cases_metadata.parquet (Shape: (100, 8), Time: 0.10s)


,case_id,case_name,split_type,fold,fault_transmissibility,porosity_multiplier,permeability_multiplier,aquifer_pore_volume
0,1,Case 1,train,0,0.052351,1.082307,3.047346,68.352306
1,2,Case 2,train,1,0.089456,1.229391,6.354067,119.991455
2,3,Case 3,train,2,0.127971,0.885781,1.480993,198.958098
3,4,Case 4,train,3,0.145503,1.364136,8.602499,158.106937
4,5,Case 5,train,4,0.111200,0.949402,5.398152,136.151463


## 2. Field Production History (Ground Truth Target)
Parses `02 Field Production.xlsx`:
1. **`Field Production` Sheet**: 41 quarterly survey observations of Oil, Gas, and Water rates plus **Water Injection Rate**.
2. **`RAMP_History`**: Actual high-resolution historical cumulative and rate curves across 1998–2008.

In [3]:
t0 = time.time()
print("Ingesting 02 Field Production.xlsx (Observed Field & RAMP_History)...")
excel_fp = pd.ExcelFile(DATA_RAW / '02 Field Production.xlsx')

target_map = {
    'Gas production cumulative': 'ramp_gas_cum_mscf',
    'Gas production rate': 'ramp_gas_rate_mscfd',
    'Oil production cumulative': 'ramp_oil_cum_stb',
    'Oil production rate': 'ramp_oil_rate_stbd',
    'Water production cumulative': 'ramp_water_cum_stb',
    'Water production rate': 'ramp_water_rate_stbd'
}

ramp_dict = {}
dates = None
for sheet_name, col_name in target_map.items():
    raw = excel_fp.parse(sheet_name)
    if dates is None:
        dates = pd.to_datetime(raw.iloc[2:, 0].values)
    # RAMP_History is in the final column
    ramp_vals = pd.to_numeric(raw.iloc[2:, -1].values, errors='coerce')
    ramp_dict[col_name] = ramp_vals

df_ramp = pd.DataFrame(ramp_dict, index=dates)
df_ramp.index.name = 'date'
df_ramp.reset_index(inplace=True)

# Parse quarterly field observation sheet
df_fp = excel_fp.parse('Field Production')
df_fp.rename(columns={
    'Date': 'date',
    'Oil production rate (STB/d)': 'obs_oil_rate_stbd',
    'Gas production rate (MSCF/d)': 'obs_gas_rate_mscfd',
    'Water production rate (STB/d)': 'obs_water_rate_stbd',
    'Water injection rate (STB/d)': 'obs_water_inj_rate_stbd'
}, inplace=True)
df_fp['date'] = pd.to_datetime(df_fp['date'])
cols_fp = ['date', 'obs_oil_rate_stbd', 'obs_gas_rate_mscfd', 'obs_water_rate_stbd', 'obs_water_inj_rate_stbd']
df_fp = df_fp[cols_fp]

# Merge observed field observations and RAMP_History
df_field = pd.merge(df_ramp, df_fp, on='date', how='left')
valid_mask = df_field[['ramp_oil_cum_stb', 'obs_oil_rate_stbd']].notnull().any(axis=1)
df_field_clean = df_field[valid_mask].sort_values('date').reset_index(drop=True)

out_field = DATA_OUT / 'field_history.parquet'
df_field_clean.to_parquet(out_field, index=False)

print(f"Saved: {out_field} (Shape: {df_field_clean.shape}, Time: {time.time()-t0:.2f}s)")
df_field_clean.head()

Ingesting 02 Field Production.xlsx (Observed Field & RAMP_History)...


Saved: /Users/tengkuanas/Projects/mwap-exdw-surrogate-model/data/field_history.parquet (Shape: (41, 11), Time: 6.64s)


,date,ramp_gas_cum_mscf,ramp_gas_rate_mscfd,ramp_oil_cum_stb,ramp_oil_rate_stbd,ramp_water_cum_stb,ramp_water_rate_stbd,obs_oil_rate_stbd,obs_gas_rate_mscfd,obs_water_rate_stbd,obs_water_inj_rate_stbd
0,1998-01-01,0.00,0.00,0.000000e+00,0.00,0.0,0.0,0.00,0.00,0.0,0.00
1,1998-04-01,0.00,0.00,0.000000e+00,0.00,0.0,0.0,0.00,0.00,0.0,0.00
2,1998-07-01,0.00,0.00,0.000000e+00,0.00,0.0,0.0,0.00,0.00,0.0,0.00
3,1998-10-01,0.00,5633.97,0.000000e+00,15507.77,0.0,0.0,15507.77,5633.97,0.0,14748.87
4,1999-01-01,518325.24,5549.46,1.426715e+06,15275.16,0.0,0.0,15275.16,5549.46,0.0,12098.46


## 3. Simulation Runs Timeseries (All 100 Cases)
Extracts all 100 simulation curves across the 6 simulation metrics:
- `gas_cum_mscf`
- `gas_rate_mscfd`
- `oil_cum_stb`
- `oil_rate_stbd`
- `water_cum_stb`
- `water_rate_stbd`

Reshapes the wide matrix into a long-format DataFrame `(case_id, date)` with flag `is_quarterly` indicating alignment with official quarterly survey timestamps.

In [4]:
t0 = time.time()
print("Ingesting all 100 simulation curves...")

quarterly_dates = set(df_fp['date'])
sheets_sim = {
    'gas_cum_mscf': 'Gas production cumulative',
    'gas_rate_mscfd': 'Gas production rate',
    'oil_cum_stb': 'Oil production cumulative',
    'oil_rate_stbd': 'Oil production rate',
    'water_cum_stb': 'Water production cumulative',
    'water_rate_stbd': 'Water production rate'
}

parsed_sheets = {k: excel_fp.parse(v) for k, v in sheets_sim.items()}
sim_dates = pd.to_datetime(parsed_sheets['gas_cum_mscf'].iloc[2:, 0].values)

case_dfs = []
for c_idx in range(1, 101):
    series_data = {}
    for metric_name, sheet_df in parsed_sheets.items():
        series_data[metric_name] = pd.to_numeric(sheet_df.iloc[2:, c_idx].values, errors='coerce')
    
    c_df = pd.DataFrame(series_data, index=sim_dates)
    c_df.dropna(how='all', inplace=True)
    c_df.index.name = 'date'
    c_df.reset_index(inplace=True)
    c_df['case_id'] = c_idx
    c_df['case_name'] = f'Case {c_idx}'
    c_df['is_quarterly'] = c_df['date'].isin(quarterly_dates)
    case_dfs.append(c_df)

df_all_sim = pd.concat(case_dfs, ignore_index=True)
cols_sim = [
    'case_id', 'case_name', 'date', 'is_quarterly',
    'gas_cum_mscf', 'gas_rate_mscfd',
    'oil_cum_stb', 'oil_rate_stbd',
    'water_cum_stb', 'water_rate_stbd'
]
df_all_sim = df_all_sim[cols_sim].sort_values(['case_id', 'date']).reset_index(drop=True)

out_sim = DATA_OUT / 'simulation_timeseries.parquet'
df_all_sim.to_parquet(out_sim, index=False)

print(f"Saved: {out_sim} (Shape: {df_all_sim.shape}, Time: {time.time()-t0:.2f}s)")
df_all_sim.head()

Ingesting all 100 simulation curves...


Saved: /Users/tengkuanas/Projects/mwap-exdw-surrogate-model/data/simulation_timeseries.parquet (Shape: (17222, 10), Time: 6.99s)


,case_id,case_name,date,is_quarterly,gas_cum_mscf,gas_rate_mscfd,oil_cum_stb,oil_rate_stbd,water_cum_stb,water_rate_stbd
0,1,Case 1,1998-01-01,True,0.0,0.0,0.0,0.0,0.0,0.0
1,1,Case 1,1998-01-02,False,0.0,0.0,0.0,0.0,0.0,0.0
2,1,Case 1,1998-01-05,False,0.0,0.0,0.0,0.0,0.0,0.0
3,1,Case 1,1998-01-14,False,0.0,0.0,0.0,0.0,0.0,0.0
4,1,Case 1,1998-02-10,False,0.0,0.0,0.0,0.0,0.0,0.0


## 4. Verification & Summary
Check output files, sizes on disk, memory footprint, and monotonic cumulative checks.

In [5]:
print("=" * 65)
print("PREPROCESSED DATASET INTEGRITY REPORT")
print("=" * 65)

for p in sorted(DATA_OUT.glob('*.parquet')):
    df = pd.read_parquet(p)
    file_kb = p.stat().st_size / 1024
    print(f"File: {p.name:<30} | Size: {file_kb:8.1f} KB | Shape: {str(df.shape):<15} | Memory: {df.memory_usage().sum()/1024:.1f} KB")

# Verification: Monotonicity check on cumulative production
sim_check = pd.read_parquet(DATA_OUT / 'simulation_timeseries.parquet')
for fluid in ['gas_cum_mscf', 'oil_cum_stb', 'water_cum_stb']:
    diffs = sim_check.groupby('case_id')[fluid].diff().dropna()
    min_diff = diffs.min()
    print(f"Monotonicity Check ({fluid}): Min step change = {min_diff:.4f} (All non-negative: {min_diff >= 0})")

print("=" * 65)
print("Data preparation complete! All 3 parquets ready for EDA and modeling.")

PREPROCESSED DATASET INTEGRITY REPORT
File: cases_metadata.parquet         | Size:      9.9 KB | Shape: (100, 8)        | Memory: 7.5 KB
File: field_history.parquet          | Size:     11.0 KB | Shape: (41, 11)        | Memory: 3.7 KB
File: simulation_timeseries.parquet  | Size:    825.0 KB | Shape: (17222, 10)     | Memory: 1344.2 KB
Monotonicity Check (gas_cum_mscf): Min step change = 0.0000 (All non-negative: True)
Monotonicity Check (oil_cum_stb): Min step change = 0.0000 (All non-negative: True)
Monotonicity Check (water_cum_stb): Min step change = 0.0000 (All non-negative: True)
Data preparation complete! All 3 parquets ready for EDA and modeling.
